In [6]:
import pandas as pd
import joblib
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [7]:
# load unnormalised dataset
FEATURE_COLUMNS = ['d_front', 'd_back', 'v_front', 'v_back']
dataset = pd.read_csv('simulation-dataset.csv')
training_datas_unnormalised = dataset[FEATURE_COLUMNS].values

unnormalised_mins = training_datas_unnormalised.min(axis=0)
unnormalised_maxs = training_datas_unnormalised.max(axis=0)

details = pd.DataFrame({
    'Features': FEATURE_COLUMNS,
    'Data Min': unnormalised_mins,
    'Data Max': unnormalised_maxs
})
# print(details.to_string())
display(details)

# load scaler file and normalise
scaler = joblib.load('scaler.pkl')
training_datas = scaler.transform(training_datas_unnormalised)
print(f"\nFirst 5 lines of dataset:")
# print(f"{training_datas[:5]}\n")

column_names = ['d_front (0)', 'd_back (1)', 'v_front (2)', 'v_back (3)']
training_data_df = pd.DataFrame(data=training_datas, columns=column_names)
display(training_data_df[:5])

LABEL_COLUMN = 'ego_a'
training_labels= dataset[LABEL_COLUMN].values
print(f"\nFirst 5 lines of labels:")
print(f"{training_labels[:5]}\n")

,Features,Data Min,Data Max
0,d_front,-0.092464,311.600923
1,d_back,-0.077204,305.828968
2,v_front,-5.563068,7.548104
3,v_back,-5.361264,7.422031



First 5 lines of dataset:


,d_front (0),d_back (1),v_front (2),v_back (3)
0,-0.444195,-0.894804,-0.183601,-0.362558
1,-0.444536,-0.894360,-0.349543,-0.128195
2,-0.445220,-0.892951,-0.350522,0.146128
3,-0.445917,-0.890531,-0.356197,0.405182
4,-0.446640,-0.887611,-0.362737,0.409462



First 5 lines of labels:
[0. 3. 3. 3. 3.]



In [8]:
# find normalise rule
print(f"Scaler type: {scaler}")
details = pd.DataFrame({
        'features': FEATURE_COLUMNS,
        'Mean': scaler.mean_,
        'SD': scaler.scale_
    })
# print(details.to_string())
display(details)

Scaler type: StandardScaler()


,features,Mean,SD
0,d_front,68.479297,52.128595
1,d_back,69.437695,52.274740
2,v_front,0.393321,2.142253
3,v_back,0.718784,1.982537


Normalisation law:
$$X=(X_{unnorm} - mean) / SD$$

---
## Custom safety properties

### Safe front

$$ pos_{front} - pos_{ego} > L $$

$$ (x_{front} - v_{front}^2 / 2B_{max}) - pos_{ego} > L $$

In metrics relative to the ego car, as per our data ($d_{front}$ is the distance from ego to front car, $v_{front}$ is the velocity difference between the two cars):

$$ d_{front} - v_{front}^2/2B_{max} > L $$

$$ d_{front} > L + v_{front}^2/2B_{max} $$


### Safe back

$$ pos_{ego} - pos_{back} > L $$

$$ pos_{ego} - (x_{back} - v_{back}^2 / 2A_{max}) > L $$

In metrics relative to the ego car::

$$ d_{back} - v_{back}^2/2A_{max} > L $$

$$ d_{back} > L + v_{back}^2/2A_{max} $$

---
## Marabou-friendly simplification ಥ_ಥ

Avoiding non-linearity by using constant values for $v_{front}$ and $v_{back}$, assuming worst-case:

$$ d_{front} > L + v_{max}^2/2B_{max} $$

$$ d_{back} > L + v_{min}^2/2A_{max} $$

---
## Final vehicle code!
```
Bmax = 5.0
Amax = 3.0
Vmin = 0.0001
Vmax = 20.0
L = 4.0

safeFront : UnnormalisedInput -> Bool
safeFront x =
  x ! distanceToFrontCar > L and
  x ! distanceToFrontCar > L + (Vmax * Vmax)/(2 * Bmax)

safeBack : UnnormalisedInput -> Bool
safeBack x = 
  x ! distanceToBackCar > L and
  x ! distanceToBackCar > L + (Vmin * Vmin)/(2 * Amax)

@property
property1 : Bool
property1 = forall x .
  validInput x and
  not (safeFront x) =>
  actionToTake brake x

@property
property2 : Bool
property2 = forall x .
  validInput x and
  not (safeBack x) =>
  actionToTake accelerate x

```

---
## Example

In [14]:
! vehicle verify \
    --specification car-safety.vcl \
    --verifier Marabou \
    --network nnModel:nn_model.onnx \
    --property property1



In order to provide support, Vehicle has automatically converted the strict inequalities to non-strict inequalites. This is not sound, but errors will be at most the floating point epsilon used by the verifier, which is usually very small (e.g. 1e-9). However, this may lead to unexpected behaviour (e.g. loss of the law of excluded middle).

See https://github.com/vehicle-lang/vehicle/issues/74 for further details.

Verifying properties:
  property1 [......................................................] 0/4 queries
    result: ✗ - Marabou found a counterexample
      x: [ 3.99997824384, 61.99136738596, 1.3847556884, 2.407334553344 ]


In [18]:
! vehicle verify \
    --specification car-safety.vcl \
    --verifier Marabou \
    --network nnModel:nn_model.onnx \
    --property property2



In order to provide support, Vehicle has automatically converted the strict inequalities to non-strict inequalites. This is not sound, but errors will be at most the floating point epsilon used by the verifier, which is usually very small (e.g. 1e-9). However, this may lead to unexpected behaviour (e.g. loss of the law of excluded middle).

See https://github.com/vehicle-lang/vehicle/issues/74 for further details.

Verifying properties:
  property2 [......................................................] 0/4 queries
    result: ✗ - Marabou found a counterexample
      x: [ 18.595619943675, 4.00001864378, 0.835370622044, 2.017575709292 ]


In [ ]:
! vehicle compile \
    --target MarabouQueries \
    --specification car-safety.vcl \
    --network nnModel:nn_model.onnx \
    --parameter epsilon:0.05
    # --output query_cache \




(property1, 1)
+y0 -y1 <= 0.0
x0 <= -1.23692758264442
x0 <= 4.665800526563204
x0 >= -1.3173529998266786
x1 <= 4.524006680855801
x1 >= -1.3317120085150114
x2 <= 3.386520172920752
x2 >= -2.8271119237550373
x3 <= 3.431586396622106
x3 >= -3.117242200271672


(property1, 2)
+y0 -y1 <= 0.0
x0 <= -0.4695944135843293
x0 <= 4.665800526563204
x0 >= -1.3173529998266786
x1 <= 4.524006680855801
x1 >= -1.3317120085150114
x2 <= 3.386520172920752
x2 >= -2.8271119237550373
x3 <= 3.431586396622106
x3 >= -3.117242200271672


(property1, 3)
+y0 -y2 <= 0.0
x0 <= -1.23692758264442
x0 <= 4.665800526563204
x0 >= -1.3173529998266786
x1 <= 4.524006680855801
x1 >= -1.3317120085150114
x2 <= 3.386520172920752
x2 >= -2.8271119237550373
x3 <= 3.431586396622106
x3 >= -3.117242200271672


(property1, 4)
+y0 -y2 <= 0.0
x0 <= -0.4695944135843293
x0 <= 4.665800526563204
x0 >= -1.3173529998266786
x1 <= 4.524006680855801
x1 >= -1.3317120085150114
x2 <= 3.386520172920752
x2 >= -2.8271119237550373
x3 <= 3.431586396622106
x